In [1]:
"""
Unified Stock Price Movement Classification Model - All 10 Stocks
==================================================================
Predicts next-day price movement direction (DOWN/FLAT/UP) for all stocks
Baseline Models: Logistic Regression, XGBoost, Random Forest, SVM
Includes Hyperparameter Tuning with GridSearchCV
DATE-BASED SPLIT: Train on earlier dates, validate on later dates
Stock encoding included as feature for model differentiation
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    f1_score,
    make_scorer
)
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*80)
print("UNIFIED STOCK PRICE MOVEMENT CLASSIFICATION - ALL 10 STOCKS")
print("DATE-BASED SPLIT with HYPERPARAMETER TUNING")
print("="*80)
print()

# ============================================================================
# 1. DATA LOADING
# ============================================================================
print("Step 1: Loading Data for All Stocks")
print("-" * 80)

# Load datasets
train_df = pd.read_csv('training_data.csv')
test_df = pd.read_csv('test_data.csv')
    
# Convert date to datetime and sort
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

train_df = train_df.sort_values(['stock', 'date']).reset_index(drop=True)
test_df = test_df.sort_values(['stock', 'date']).reset_index(drop=True)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\nTraining date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Test date range: {test_df['date'].min()} to {test_df['date'].max()}")
print()

# Display stock distribution
print("Stock Distribution in Training Data:")
stock_dist = train_df['stock'].value_counts().sort_index()
print(stock_dist)
print()

# Check class distribution
print("Class Distribution in Training Data:")
class_dist = train_df['target'].value_counts().sort_index()
print(class_dist)
print(f"\nClass proportions:")
print((class_dist / len(train_df)).round(3))
print()

# Class distribution per stock
print("Class Distribution by Stock:")
print(train_df.groupby(['stock', 'target']).size().unstack(fill_value=0))
print()

UNIFIED STOCK PRICE MOVEMENT CLASSIFICATION - ALL 10 STOCKS
DATE-BASED SPLIT with HYPERPARAMETER TUNING

Step 1: Loading Data for All Stocks
--------------------------------------------------------------------------------
Training data shape: (7736, 23)
Test data shape: (447, 23)

Training date range: 2023-01-03 00:00:00 to 2025-12-26 00:00:00
Test date range: 2026-01-05 00:00:00 to 2026-02-27 00:00:00

Stock Distribution in Training Data:
stock
AP       757
AREIT    773
CNVRG    763
DMC      781
JFC      798
MBT      743
SCC      745
SM       764
SMPH     811
TEL      801
Name: count, dtype: int64

Class Distribution in Training Data:
target
DOWN    1708
FLAT    4344
UP      1684
Name: count, dtype: int64

Class proportions:
target
DOWN    0.221
FLAT    0.562
UP      0.218
Name: count, dtype: float64

Class Distribution by Stock:
target  DOWN  FLAT   UP
stock                  
AP       149   452  156
AREIT    107   543  123
CNVRG    236   293  234
DMC      158   459  164
JFC      189 

In [4]:
# ============================================================================
# 2. FEATURE ENGINEERING & PREPROCESSING
# ============================================================================
print("Step 2: Feature Engineering & Preprocessing")
print("-" * 80)

# One-hot encode the 'stock' column
print("One-hot encoding 'stock' column...")
train_encoded = pd.get_dummies(train_df, columns=['stock'], prefix='stock', drop_first=False)
test_encoded = pd.get_dummies(test_df, columns=['stock'], prefix='stock', drop_first=False)

# Ensure test set has all stock columns
train_stock_cols = [col for col in train_encoded.columns if col.startswith('stock_')]
test_stock_cols = [col for col in test_encoded.columns if col.startswith('stock_')]

for col in train_stock_cols:
    if col not in test_stock_cols:
        test_encoded[col] = 0

# Align column order
test_encoded = test_encoded[train_encoded.columns]

print(f"Stock columns created: {len(train_stock_cols)}")
print(f"Stock tickers: {sorted([col.replace('stock_', '') for col in train_stock_cols])}")
print()

# Define columns to drop
cols_to_drop = ['date', 'next_close', 'price_change_pct', 'target']

# Separate features and target
X_train_full = train_encoded.drop(cols_to_drop + ['target_encoded'], axis=1)
y_train_full = train_encoded['target_encoded']
dates_train = train_df['date']

X_test = test_encoded.drop(cols_to_drop + ['target_encoded'], axis=1)
y_test = test_encoded['target_encoded']
dates_test = test_df['date']

print(f"Feature matrix shape (full training): {X_train_full.shape}")
print(f"Feature matrix shape (test): {X_test.shape}")
print(f"Number of features: {X_train_full.shape[1]}")
print()

Step 2: Feature Engineering & Preprocessing
--------------------------------------------------------------------------------
One-hot encoding 'stock' column...
Stock columns created: 10
Stock tickers: ['AP', 'AREIT', 'CNVRG', 'DMC', 'JFC', 'MBT', 'SCC', 'SM', 'SMPH', 'TEL']

Feature matrix shape (full training): (7736, 27)
Feature matrix shape (test): (447, 27)
Number of features: 27



In [5]:
# ============================================================================
# 3. DATE-BASED TRAIN/VALIDATION SPLIT
# ============================================================================
print("Step 3: Creating Date-Based Train/Validation Split")
print("-" * 80)

# Calculate split index (80% train, 20% validation by date)
split_idx = int(len(X_train_full) * 0.8)

# Split based on temporal order
X_train = X_train_full.iloc[:split_idx]
X_val = X_train_full.iloc[split_idx:]
y_train = y_train_full.iloc[:split_idx]
y_val = y_train_full.iloc[split_idx:]
dates_train_split = dates_train.iloc[:split_idx]
dates_val_split = dates_train.iloc[split_idx:]

print(f"Training set size: {len(X_train)} ({len(X_train)/len(X_train_full)*100:.1f}%)")
print(f"Validation set size: {len(X_val)} ({len(X_val)/len(X_train_full)*100:.1f}%)")
print(f"Test set size: {len(X_test)}")
print()

print(f"Training date range: {dates_train_split.min()} to {dates_train_split.max()}")
print(f"Validation date range: {dates_val_split.min()} to {dates_val_split.max()}")
print(f"Test date range: {dates_test.min()} to {dates_test.max()}")
print()

# Check class distribution after split
print("Class distribution after date-based split:")
train_class_dist = y_train.value_counts().sort_index()
val_class_dist = y_val.value_counts().sort_index()
test_class_dist = y_test.value_counts().sort_index()

print(f"Training: {train_class_dist.to_dict()}")
print(f"  Proportions: {(train_class_dist / len(y_train)).round(3).to_dict()}")
print(f"Validation: {val_class_dist.to_dict()}")
print(f"  Proportions: {(val_class_dist / len(y_val)).round(3).to_dict()}")
print(f"Test: {test_class_dist.to_dict()}")
print(f"  Proportions: {(test_class_dist / len(y_test)).round(3).to_dict()}")
print()

Step 3: Creating Date-Based Train/Validation Split
--------------------------------------------------------------------------------
Training set size: 6188 (80.0%)
Validation set size: 1548 (20.0%)
Test set size: 447

Training date range: 2023-01-03 00:00:00 to 2025-12-26 00:00:00
Validation date range: 2023-01-03 00:00:00 to 2025-12-26 00:00:00
Test date range: 2026-01-05 00:00:00 to 2026-02-27 00:00:00

Class distribution after date-based split:
Training: {0.0: 1330, 1.0: 3531, 2.0: 1327}
  Proportions: {0.0: 0.215, 1.0: 0.571, 2.0: 0.214}
Validation: {0.0: 378, 1.0: 813, 2.0: 357}
  Proportions: {0.0: 0.244, 1.0: 0.525, 2.0: 0.231}
Test: {0.0: 83, 1.0: 269, 2.0: 95}
  Proportions: {0.0: 0.186, 1.0: 0.602, 2.0: 0.213}



In [6]:
# ============================================================================
# 4. FEATURE SCALING
# ============================================================================
print("Step 4: Feature Scaling")
print("-" * 80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_train_full_scaled = scaler.fit_transform(X_train_full)

print("Features scaled using StandardScaler (fitted on training set only)")
print(f"Scaler mean shape: {scaler.mean_.shape}")
print(f"Scaler scale shape: {scaler.scale_.shape}")
print()

Step 4: Feature Scaling
--------------------------------------------------------------------------------
Features scaled using StandardScaler (fitted on training set only)
Scaler mean shape: (27,)
Scaler scale shape: (27,)



In [7]:
# ============================================================================
# 5. HYPERPARAMETER TUNING
# ============================================================================
print("Step 5: Hyperparameter Tuning with GridSearchCV")
print("=" * 80)
print("Note: This may take some time due to the large dataset and parameter space...")
print()

# Time Series Cross-Validation for tuning
tscv = TimeSeriesSplit(n_splits=3)

# Scoring metric
f1_scorer = make_scorer(f1_score, average='weighted')

tuned_models = {}
best_params_dict = {}

# --- 1. Logistic Regression Tuning ---
print("\n[1/4] Tuning Logistic Regression")
print("-" * 80)

lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [1000]
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    lr_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
lr_grid.fit(X_train_scaled, y_train)
best_params_dict['Logistic Regression'] = lr_grid.best_params_
tuned_models['Logistic Regression'] = lr_grid.best_estimator_

print(f"✓ Best parameters: {lr_grid.best_params_}")
print(f"✓ Best CV score (F1): {lr_grid.best_score_:.4f}")

# --- 2. XGBoost Tuning ---
print("\n[2/4] Tuning XGBoost")
print("-" * 80)

xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss'),
    xgb_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
xgb_grid.fit(X_train_scaled, y_train)
best_params_dict['XGBoost'] = xgb_grid.best_params_
tuned_models['XGBoost'] = xgb_grid.best_estimator_

print(f"✓ Best parameters: {xgb_grid.best_params_}")
print(f"✓ Best CV score (F1): {xgb_grid.best_score_:.4f}")

# --- 3. Random Forest Tuning ---
print("\n[3/4] Tuning Random Forest")
print("-" * 80)

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
rf_grid.fit(X_train_scaled, y_train)
best_params_dict['Random Forest'] = rf_grid.best_params_
tuned_models['Random Forest'] = rf_grid.best_estimator_

print(f"✓ Best parameters: {rf_grid.best_params_}")
print(f"✓ Best CV score (F1): {rf_grid.best_score_:.4f}")

# --- 4. SVM Tuning ---
print("\n[4/4] Tuning Support Vector Machine (SVM)")
print("-" * 80)

svm_param_grid = {
    'C': [0.1, 1.0, 10.0],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

svm_grid = GridSearchCV(
    SVC(random_state=RANDOM_STATE),
    svm_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
svm_grid.fit(X_train_scaled, y_train)
best_params_dict['SVM'] = svm_grid.best_params_
tuned_models['SVM'] = svm_grid.best_estimator_

print(f"✓ Best parameters: {svm_grid.best_params_}")
print(f"✓ Best CV score (F1): {svm_grid.best_score_:.4f}")

Step 5: Hyperparameter Tuning with GridSearchCV
Note: This may take some time due to the large dataset and parameter space...


[1/4] Tuning Logistic Regression
--------------------------------------------------------------------------------
Starting grid search...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
✓ Best parameters: {'C': 0.01, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
✓ Best CV score (F1): 0.4105

[2/4] Tuning XGBoost
--------------------------------------------------------------------------------
Starting grid search...
Fitting 3 folds for each of 108 candidates, totalling 324 fits
✓ Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}
✓ Best CV score (F1): 0.4285

[3/4] Tuning Random Forest
--------------------------------------------------------------------------------
Starting grid search...
Fitting 3 folds for each of 216 candidates, totalling 648 fits
✓ Best parameters: {'max

In [8]:
# ============================================================================
# 6. MODEL EVALUATION WITH TUNED PARAMETERS
# ============================================================================
print("\n" + "="*80)
print("Step 6: Evaluating Tuned Models")
print("="*80)

results = {
    'model': [],
    'train_accuracy': [],
    'val_accuracy': [],
    'test_accuracy': [],
    'train_f1_weighted': [],
    'val_f1_weighted': [],
    'test_f1_weighted': [],
    'best_params': []
}

predictions = {}

for model_name, model in tuned_models.items():
    print(f"\n{model_name}")
    print("-" * 80)
    
    # Predictions
    train_pred = model.predict(X_train_scaled)
    val_pred = model.predict(X_val_scaled)
    test_pred = model.predict(X_test_scaled)
    
    predictions[model_name] = (train_pred, val_pred, test_pred)
    
    # Metrics
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    train_f1 = f1_score(y_train, train_pred, average='weighted')
    val_f1 = f1_score(y_val, val_pred, average='weighted')
    test_f1 = f1_score(y_test, test_pred, average='weighted')
    
    print(f"Train - Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   - Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print(f"Test  - Acc: {test_acc:.4f} | F1: {test_f1:.4f}")
    
    results['model'].append(model_name)
    results['train_accuracy'].append(train_acc)
    results['val_accuracy'].append(val_acc)
    results['test_accuracy'].append(test_acc)
    results['train_f1_weighted'].append(train_f1)
    results['val_f1_weighted'].append(val_f1)
    results['test_f1_weighted'].append(test_f1)
    results['best_params'].append(str(best_params_dict[model_name]))


Step 6: Evaluating Tuned Models

Logistic Regression
--------------------------------------------------------------------------------
Train - Acc: 0.5727 | F1: 0.4335
Val   - Acc: 0.5006 | F1: 0.4057
Test  - Acc: 0.6130 | F1: 0.4808

XGBoost
--------------------------------------------------------------------------------
Train - Acc: 0.5858 | F1: 0.4603
Val   - Acc: 0.5239 | F1: 0.3637
Test  - Acc: 0.6018 | F1: 0.4651

Random Forest
--------------------------------------------------------------------------------
Train - Acc: 0.5853 | F1: 0.4539
Val   - Acc: 0.5252 | F1: 0.3617
Test  - Acc: 0.6063 | F1: 0.4664

SVM
--------------------------------------------------------------------------------
Train - Acc: 0.5827 | F1: 0.4547
Val   - Acc: 0.3960 | F1: 0.3523
Test  - Acc: 0.5615 | F1: 0.4564


In [9]:
# ============================================================================
# 7. TIME SERIES CROSS-VALIDATION ON FULL TRAINING SET
# ============================================================================
print("\n" + "="*80)
print("Step 7: Time Series Cross-Validation (5 Splits)")
print("="*80)

cv_strategy = TimeSeriesSplit(n_splits=5)
cv_results = {}

for model_name, model in tuned_models.items():
    print(f"\n{model_name}:")
    cv_accuracy = cross_val_score(model, X_train_full_scaled, y_train_full, 
                                   cv=cv_strategy, scoring='accuracy', n_jobs=-1)
    cv_f1 = cross_val_score(model, X_train_full_scaled, y_train_full, 
                            cv=cv_strategy, scoring='f1_weighted', n_jobs=-1)
    
    cv_results[model_name] = {
        'accuracy': cv_accuracy,
        'f1': cv_f1
    }
    
    print(f"  CV Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
    print(f"  CV F1: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
    print(f"  Fold scores: {[f'{score:.4f}' for score in cv_accuracy]}")


Step 7: Time Series Cross-Validation (5 Splits)

Logistic Regression:
  CV Accuracy: 0.5446 (+/- 0.0353)
  CV F1: 0.4012 (+/- 0.0342)
  Fold scores: ['0.4911', '0.5687', '0.5749', '0.5749', '0.5136']

XGBoost:
  CV Accuracy: 0.5213 (+/- 0.0549)
  CV F1: 0.4105 (+/- 0.0330)
  Fold scores: ['0.4174', '0.5578', '0.5741', '0.5252', '0.5322']

Random Forest:
  CV Accuracy: 0.5192 (+/- 0.0592)
  CV F1: 0.4020 (+/- 0.0267)
  Fold scores: ['0.4143', '0.5004', '0.5756', '0.5718', '0.5337']

SVM:
  CV Accuracy: 0.5375 (+/- 0.0369)
  CV F1: 0.3919 (+/- 0.0379)
  Fold scores: ['0.4810', '0.5625', '0.5663', '0.5718', '0.5058']


In [10]:
# ============================================================================
# 8. DETAILED EVALUATION ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("Step 8: Detailed Evaluation on Test Set (All Stocks)")
print("="*80)

class_names = ['DOWN (0)', 'FLAT (1)', 'UP (2)']

for model_name in ['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']:
    print(f"\n{model_name}")
    print("-" * 80)
    
    test_pred = predictions[model_name][2]
    
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred, target_names=class_names, zero_division=0))
    
    cm = confusion_matrix(y_test, test_pred)
    print("\nConfusion Matrix:")
    print(cm)


Step 8: Detailed Evaluation on Test Set (All Stocks)

Logistic Regression
--------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

    DOWN (0)       0.00      0.00      0.00        83
    FLAT (1)       0.61      1.00      0.76       269
      UP (2)       0.67      0.06      0.12        95

    accuracy                           0.61       447
   macro avg       0.43      0.35      0.29       447
weighted avg       0.51      0.61      0.48       447


Confusion Matrix:
[[  0  81   2]
 [  0 268   1]
 [  0  89   6]]

XGBoost
--------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

    DOWN (0)       0.00      0.00      0.00        83
    FLAT (1)       0.61      0.99      0.75       269
      UP (2)       0.33      0.03      0.06        95

    accuracy                           0.6

In [11]:
# ============================================================================
# 9. PER-STOCK PERFORMANCE ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("Step 9: Per-Stock Performance Analysis")
print("="*80)

# Add stock column back for analysis
test_df_copy = test_df.copy()
test_df_copy['actual'] = y_test.values

per_stock_results = []

for model_name in ['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']:
    test_df_copy[f'pred_{model_name}'] = predictions[model_name][2]
    
    for stock in sorted(test_df_copy['stock'].unique()):
        stock_mask = test_df_copy['stock'] == stock
        y_stock_true = test_df_copy.loc[stock_mask, 'actual']
        y_stock_pred = test_df_copy.loc[stock_mask, f'pred_{model_name}']
        
        if len(y_stock_true) > 0:
            acc = accuracy_score(y_stock_true, y_stock_pred)
            f1 = f1_score(y_stock_true, y_stock_pred, average='weighted', zero_division=0)
            
            per_stock_results.append({
                'model': model_name,
                'stock': stock,
                'accuracy': acc,
                'f1_weighted': f1,
                'n_samples': len(y_stock_true)
            })

per_stock_df = pd.DataFrame(per_stock_results)

print("\nPer-Stock Test Accuracy:")
pivot_acc = per_stock_df.pivot(index='stock', columns='model', values='accuracy')
print(pivot_acc.to_string())

print("\nPer-Stock Test F1 Score:")
pivot_f1 = per_stock_df.pivot(index='stock', columns='model', values='f1_weighted')
print(pivot_f1.to_string())

print("\nSample counts per stock:")
print(per_stock_df.groupby('stock')['n_samples'].first().to_string())


Step 9: Per-Stock Performance Analysis

Per-Stock Test Accuracy:
model  Logistic Regression  Random Forest       SVM   XGBoost
stock                                                        
AP                0.804878       0.804878  0.804878  0.804878
AREIT             0.789474       0.789474  0.789474  0.789474
CNVRG             0.333333       0.333333  0.307692  0.307692
DMC               0.487179       0.435897  0.435897  0.435897
JFC               0.650000       0.650000  0.583333  0.650000
MBT               0.500000       0.500000  0.500000  0.500000
SCC               0.525000       0.500000  0.500000  0.475000
SM                0.695652       0.695652  0.673913  0.695652
SMPH              0.625000       0.625000  0.589286  0.625000
TEL               0.673913       0.673913  0.413043  0.673913

Per-Stock Test F1 Score:
model  Logistic Regression  Random Forest       SVM   XGBoost
stock                                                        
AP                0.717864       0.71786

In [12]:
# ============================================================================
# 10. VISUALIZE CONFUSION MATRICES
# ============================================================================
print("\n" + "="*80)
print("Step 10: Creating Confusion Matrix Visualizations")
print("="*80)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, model_name in enumerate(['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']):
    val_pred = predictions[model_name][1]
    test_pred = predictions[model_name][2]
    
    # Validation confusion matrix
    cm_val = confusion_matrix(y_val, val_pred)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['DOWN', 'FLAT', 'UP'],
                yticklabels=['DOWN', 'FLAT', 'UP'],
                ax=axes[0, idx], cbar=False)
    axes[0, idx].set_title(f'{model_name}\nValidation Set (All Stocks)', fontsize=10)
    axes[0, idx].set_ylabel('True Label')
    axes[0, idx].set_xlabel('Predicted Label')
    
    # Test confusion matrix
    cm_test = confusion_matrix(y_test, test_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
                xticklabels=['DOWN', 'FLAT', 'UP'],
                yticklabels=['DOWN', 'FLAT', 'UP'],
                ax=axes[1, idx], cbar=False)
    axes[1, idx].set_title(f'{model_name}\nTest Set (All Stocks)', fontsize=10)
    axes[1, idx].set_ylabel('True Label')
    axes[1, idx].set_xlabel('Predicted Label')

plt.tight_layout()
output_dir = 'models_unified'
os.makedirs(output_dir, exist_ok=True)
plt.savefig(f'{output_dir}/confusion_matrices_unified.png', dpi=300, bbox_inches='tight')
print(f"✓ Confusion matrices saved as '{output_dir}/confusion_matrices_unified.png'")
plt.close()


Step 10: Creating Confusion Matrix Visualizations
✓ Confusion matrices saved as 'models_unified/confusion_matrices_unified.png'


In [13]:
# ============================================================================
# 11. RESULTS SUMMARY
# ============================================================================
print("\n" + "="*80)
print("Step 11: Model Comparison Summary")
print("="*80)

results_df = pd.DataFrame(results)
print("\nOverall Performance Comparison (All 10 Stocks):")
print(results_df[['model', 'train_accuracy', 'val_accuracy', 'test_accuracy', 
                  'train_f1_weighted', 'val_f1_weighted', 'test_f1_weighted']].to_string(index=False))

print("\n\nBest Hyperparameters Found:")
print("-" * 80)
for model_name, params in best_params_dict.items():
    print(f"\n{model_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")

best_idx = results_df['test_accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'model']
best_test_acc = results_df.loc[best_idx, 'test_accuracy']
best_test_f1 = results_df.loc[best_idx, 'test_f1_weighted']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test Accuracy: {best_test_acc:.4f}")
print(f"   Test F1: {best_test_f1:.4f}")


Step 11: Model Comparison Summary

Overall Performance Comparison (All 10 Stocks):
              model  train_accuracy  val_accuracy  test_accuracy  train_f1_weighted  val_f1_weighted  test_f1_weighted
Logistic Regression        0.572721      0.500646       0.612975           0.433471         0.405658          0.480759
            XGBoost        0.585811      0.523902       0.601790           0.460288         0.363682          0.465093
      Random Forest        0.585326      0.525194       0.606264           0.453903         0.361696          0.466422
                SVM        0.582741      0.395995       0.561521           0.454706         0.352273          0.456359


Best Hyperparameters Found:
--------------------------------------------------------------------------------

Logistic Regression:
  C: 0.01
  max_iter: 1000
  penalty: l2
  solver: lbfgs

XGBoost:
  colsample_bytree: 1.0
  learning_rate: 0.1
  max_depth: 3
  n_estimators: 50
  subsample: 1.0

Random Forest:
  max_dep

In [14]:
# ============================================================================
# 12. SAVE MODELS AND ARTIFACTS
# ============================================================================
print("\n" + "="*80)
print("Step 12: Saving Unified Models and Artifacts")
print("="*80)

# Save tuned models
joblib.dump(tuned_models['Logistic Regression'], f'{output_dir}/logistic_regression_unified_tuned.pkl')
joblib.dump(tuned_models['XGBoost'], f'{output_dir}/xgboost_unified_tuned.pkl')
joblib.dump(tuned_models['Random Forest'], f'{output_dir}/random_forest_unified_tuned.pkl')
joblib.dump(tuned_models['SVM'], f'{output_dir}/svm_unified_tuned.pkl')
joblib.dump(scaler, f'{output_dir}/scaler_unified.pkl')

# Save best parameters
joblib.dump(best_params_dict, f'{output_dir}/best_params_unified.pkl')

# Save feature info
feature_info = {
    'feature_names': X_train_full.columns.tolist(),
    'stock_columns': train_stock_cols,
    'n_features': X_train_full.shape[1],
    'split_method': 'date_based',
    'train_date_range': (str(dates_train_split.min()), str(dates_train_split.max())),
    'val_date_range': (str(dates_val_split.min()), str(dates_val_split.max())),
    'test_date_range': (str(dates_test.min()), str(dates_test.max())),
    'stocks': sorted([col.replace('stock_', '') for col in train_stock_cols]),
    'tuning_method': 'GridSearchCV with TimeSeriesSplit',
    'unified_model': True
}
joblib.dump(feature_info, f'{output_dir}/feature_info_unified.pkl')

# Save results
results_df.to_csv(f'{output_dir}/model_comparison_unified.csv', index=False)
per_stock_df.to_csv(f'{output_dir}/per_stock_performance.csv', index=False)

# Save CV results
cv_results_df = pd.DataFrame([
    {
        'model': model_name,
        'cv_accuracy_mean': cv_results[model_name]['accuracy'].mean(),
        'cv_accuracy_std': cv_results[model_name]['accuracy'].std(),
        'cv_f1_mean': cv_results[model_name]['f1'].mean(),
        'cv_f1_std': cv_results[model_name]['f1'].std()
    }
    for model_name in tuned_models.keys()
])
cv_results_df.to_csv(f'{output_dir}/cross_validation_results.csv', index=False)

print(f"\n✓ All unified models saved in: {output_dir}/")
print(f"\n✓ Model files: *_unified_tuned.pkl")
print(f"✓ Best parameters: best_params_unified.pkl")
print(f"✓ Scaler: scaler_unified.pkl")
print(f"✓ Overall results: model_comparison_unified.csv")
print(f"✓ Per-stock performance: per_stock_performance.csv")
print(f"✓ Cross-validation results: cross_validation_results.csv")

print(f"\n" + "="*80)
print("UNIFIED MODEL TRAINING COMPLETE FOR ALL 10 STOCKS!")
print("="*80)
print(f"\nDataset Summary:")
print(f"  Total samples: {len(X_train_full)}")
print(f"  Training: {len(X_train)} samples")
print(f"  Validation: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")
print(f"  Features: {X_train_full.shape[1]} (including {len(train_stock_cols)} stock encodings)")
print(f"  Stocks: {len(train_stock_cols)}")
print(f"  Classes: 3 (DOWN/FLAT/UP)")
print("="*80)


Step 12: Saving Unified Models and Artifacts

✓ All unified models saved in: models_unified/

✓ Model files: *_unified_tuned.pkl
✓ Best parameters: best_params_unified.pkl
✓ Scaler: scaler_unified.pkl
✓ Overall results: model_comparison_unified.csv
✓ Per-stock performance: per_stock_performance.csv
✓ Cross-validation results: cross_validation_results.csv

UNIFIED MODEL TRAINING COMPLETE FOR ALL 10 STOCKS!

Dataset Summary:
  Total samples: 7736
  Training: 6188 samples
  Validation: 1548 samples
  Test: 447 samples
  Features: 27 (including 10 stock encodings)
  Stocks: 10
  Classes: 3 (DOWN/FLAT/UP)
